In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

np.random.seed(42)

DATA_DIR = Path("data")

In [2]:
def load_seeds_lookup():
    seeds_m = pd.read_csv(DATA_DIR / 'MNCAATourneySeeds.csv')
    seeds_w = pd.read_csv(DATA_DIR / 'WNCAATourneySeeds.csv')
    seeds = pd.concat([seeds_m, seeds_w], ignore_index=True)
    seeds['SeedNum'] = seeds['Seed'].str.extract(r'(\d+)').astype(float)
    seeds['SeedNorm'] = (17 - seeds['SeedNum']) / 16
    return seeds.set_index(['Season', 'TeamID'])['SeedNorm'].to_dict()

def load_raw_seeds():
    seeds_m = pd.read_csv(DATA_DIR / 'MNCAATourneySeeds.csv')
    seeds_w = pd.read_csv(DATA_DIR / 'WNCAATourneySeeds.csv')
    seeds = pd.concat([seeds_m, seeds_w], ignore_index=True)
    seeds['SeedNum'] = seeds['Seed'].str.extract(r'(\d+)').astype(int)
    return seeds.set_index(['Season', 'TeamID'])['SeedNum'].to_dict()

def _build_seed_prior_from_results(results, raw_seeds, prior_strength=5):
    records = []
    for _, r in results.iterrows():
        s = int(r['Season'])
        w_seed = raw_seeds.get((s, int(r['WTeamID'])))
        l_seed = raw_seeds.get((s, int(r['LTeamID'])))
        if w_seed and l_seed:
            records.append((min(w_seed, l_seed), max(w_seed, l_seed),
                           1 if w_seed <= l_seed else 0))
    df = pd.DataFrame(records, columns=['seed_lo', 'seed_hi', 'lo_wins'])
    agg = df.groupby(['seed_lo', 'seed_hi'])['lo_wins'].agg(['sum', 'count']).reset_index()
    agg['win_rate'] = (agg['sum'] + prior_strength * 0.5) / (agg['count'] + prior_strength)
    prior = {}
    for _, row in agg.iterrows():
        s1, s2 = int(row['seed_lo']), int(row['seed_hi'])
        prior[(s1, s2)] = row['win_rate']
        prior[(s2, s1)] = 1 - row['win_rate']
    for s in range(1, 17):
        prior[(s, s)] = 0.5
    return prior, len(records)

def compute_seed_matchup_priors():
    raw_seeds = load_raw_seeds()
    results_m = pd.read_csv(DATA_DIR / 'MNCAATourneyCompactResults.csv')
    results_w = pd.read_csv(DATA_DIR / 'WNCAATourneyCompactResults.csv')
    raw_seeds_m = {k: v for k, v in raw_seeds.items() if k[1] < 3000}
    raw_seeds_w = {k: v for k, v in raw_seeds.items() if k[1] >= 3000}
    prior_m, n_m = _build_seed_prior_from_results(results_m, raw_seeds_m, prior_strength=5)
    prior_w, n_w = _build_seed_prior_from_results(results_w, raw_seeds_w, prior_strength=5)
    print(f"  Seed priors: Men={len(prior_m)} pairs/{n_m} games, Women={len(prior_w)} pairs/{n_w} games")
    for s1, s2 in [(1, 16), (1, 8), (5, 12), (6, 11), (8, 9)]:
        print(f"    {s1}v{s2}: M={prior_m.get((s1,s2),0.5):.3f}, W={prior_w.get((s1,s2),0.5):.3f}")
    return prior_m, prior_w

def compute_sos(gender='M'):
    prefix = 'm' if gender == 'M' else 'w'
    sf = pd.read_csv(f'season_features_{prefix}.csv')
    if gender == 'M':
        results = pd.read_csv(DATA_DIR / 'MRegularSeasonCompactResults.csv')
        results = results[results['Season'] >= 2003]
    else:
        results = pd.read_csv(DATA_DIR / 'WRegularSeasonCompactResults.csv')
        results = results[results['Season'] >= 2010]
    elo_lookup = sf.set_index(['Season', 'TeamID'])['recent_elo'].to_dict()
    records = []
    for season in sf['Season'].unique():
        sg = results[results['Season'] == season]
        teams = sf[sf['Season'] == season]['TeamID'].unique()
        for team in teams:
            opps = np.concatenate([
                sg[sg['WTeamID'] == team]['LTeamID'].values,
                sg[sg['LTeamID'] == team]['WTeamID'].values
            ])
            if len(opps) > 0:
                opp_elos = [elo_lookup.get((season, o), 1500) for o in opps]
                records.append({'Season': season, 'TeamID': team,
                                'sos_mean': np.mean(opp_elos), 'sos_max': np.max(opp_elos)})
            else:
                records.append({'Season': season, 'TeamID': team, 'sos_mean': 1500, 'sos_max': 1500})
    return pd.DataFrame(records)

def load_massey_composite():
    m1 = pd.read_csv(DATA_DIR / 'MMasseyOrdinals_part_1.csv')
    m2 = pd.read_csv(DATA_DIR / 'MMasseyOrdinals_part_2.csv')
    massey = pd.concat([m1, m2], ignore_index=True)
    top_sys = ['POM', 'SAG', 'MOR', 'COL', 'DOL', 'WLK', 'AP', 'USA']
    massey = massey[massey['SystemName'].isin(top_sys)]
    idx = massey.groupby(['Season', 'SystemName', 'TeamID'])['RankingDayNum'].idxmax()
    latest = massey.loc[idx]
    comp = (latest.groupby(['Season', 'TeamID'])['OrdinalRank']
            .agg(['mean', 'min']).reset_index())
    comp.columns = ['Season', 'TeamID', 'massey_avg_rank', 'massey_best_rank']
    for col in ['massey_avg_rank', 'massey_best_rank']:
        comp[col + '_norm'] = 1 - (comp[col] / 365)
    return comp[['Season', 'TeamID', 'massey_avg_rank_norm', 'massey_best_rank_norm']]

print("Computing SOS...")
sos_m = compute_sos('M')
sos_w = compute_sos('W')
print("Loading Massey composite (men only)...")
massey_comp = load_massey_composite()
seeds_lookup = load_seeds_lookup()
raw_seeds_lookup = load_raw_seeds()
print("Computing seed matchup priors...")
seed_priors_m, seed_priors_w = compute_seed_matchup_priors()
print("Done.")

Computing SOS...
Loading Massey composite (men only)...
Computing seed matchup priors...
  Seed priors: Men=162 pairs/2585 games, Women=108 pairs/1717 games
    1v16: M=0.973, W=0.969
    1v8: M=0.771, W=0.926
    5v12: M=0.639, W=0.783
    6v11: M=0.609, W=0.668
    8v9: M=0.482, W=0.518
Done.


In [3]:
def enrich_tourney_data(df, gender='M'):
    sos = sos_m if gender == 'M' else sos_w
    seed_priors = seed_priors_m if gender == 'M' else seed_priors_w
    
    df = df.merge(sos, left_on=['Season', '1TeamID'], right_on=['Season', 'TeamID'], how='left')
    df = df.drop(columns=['TeamID'], errors='ignore')
    
    df = df.merge(sos, left_on=['Season', '2TeamID'], right_on=['Season', 'TeamID'],
                  how='left', suffixes=('', '_2'))
    df = df.drop(columns=['TeamID'], errors='ignore')
    
    for col in ['sos_mean', 'sos_max', 'sos_mean_2', 'sos_max_2']:
        if col in df.columns:
            df[col] = df[col].fillna(1500)
    
    if gender == 'M':
        df = df.merge(massey_comp, left_on=['Season', '1TeamID'],
                      right_on=['Season', 'TeamID'], how='left')
        df = df.drop(columns=['TeamID'], errors='ignore')
        df = df.merge(massey_comp, left_on=['Season', '2TeamID'],
                      right_on=['Season', 'TeamID'], how='left', suffixes=('', '_2'))
        df = df.drop(columns=['TeamID'], errors='ignore')
        
        for col in ['massey_avg_rank_norm', 'massey_best_rank_norm',
                     'massey_avg_rank_norm_2', 'massey_best_rank_norm_2']:
            if col in df.columns:
                df[col] = df[col].fillna(0)
    
    def get_seed_prior(row):
        s1 = raw_seeds_lookup.get((int(row['Season']), int(row['1TeamID'])))
        s2 = raw_seeds_lookup.get((int(row['Season']), int(row['2TeamID'])))
        if s1 and s2:
            return seed_priors.get((s1, s2), 0.5)
        return 0.5
    
    df['seed_matchup_prior'] = df.apply(get_seed_prior, axis=1)
    
    return df

In [4]:
DIFF_FEATURES = [
    'win_percentage', 'avg_points_diff', 'net_rating',
    'offensive_eff', 'defensive_eff',
    'eff_field_goal_percentage', 'true_shooting_percentage',
    'three_points_attempt_rate',
    'turnover_percentage', 'offensive_rebound_percentage', 'def_reb_perc',
    'free_throws_rate',
    'block_rate', 'steal_rate', 'assist_rate',
    'tempo',
    'neutral_win_pct', 'neutral_point_diff',
    'neutral_point_diff_smothered', 'neutral_win_pct_smothered',
    'win_pct_last5', 'win_pct_last10',
    'point_diff_last5', 'point_diff_last10',
    'off_rating_last5', 'off_rating_last10',
    'def_rating_last5', 'def_rating_last10',
    'eFG_last5', 'eFG_last10',
    'net_rating_last5', 'net_rating_last10',
    'trend_last5', 'trend_last10',
    'recent_elo', 'recent_elo_mov',
    'Seed',
    'close_game_winrate', 'blowout_rate',
    'foul_rate',
    'sos_mean', 'sos_max',
]

MASSEY_DIFF_FEATURES = ['massey_avg_rank_norm', 'massey_best_rank_norm']

def prepare_features(df, feature_list=DIFF_FEATURES, has_massey=False):
    X = pd.DataFrame()

    all_feats = feature_list.copy()
    if has_massey:
        all_feats += MASSEY_DIFF_FEATURES

    for feat in all_feats:
        col1, col2 = feat, feat + '_2'
        if col1 in df.columns and col2 in df.columns:
            X[f'diff_{feat}'] = df[col1] - df[col2]

    if 'Seed' in df.columns:
        X['seed_1'] = df['Seed']
        X['seed_2'] = df['Seed_2']
        X['seed_product'] = df['Seed'] * df['Seed_2']
        X['seed_sum'] = df['Seed'] + df['Seed_2']
    if 'recent_elo' in df.columns:
        X['elo_1'] = df['recent_elo']
        X['elo_2'] = df['recent_elo_2']
    if 'recent_elo_mov' in df.columns:
        X['elo_mov_1'] = df['recent_elo_mov']
        X['elo_mov_2'] = df['recent_elo_mov_2']

    if 'offensive_eff' in df.columns:
        X['off_vs_def_1'] = df['offensive_eff'] - df['defensive_eff_2']
        X['off_vs_def_2'] = df['offensive_eff_2'] - df['defensive_eff']
        X['matchup_diff'] = X['off_vs_def_1'] - X['off_vs_def_2']

    if 'tempo' in df.columns:
        X['tempo_avg'] = (df['tempo'] + df['tempo_2']) / 2
        X['tempo_mismatch'] = abs(df['tempo'] - df['tempo_2'])

    if 'win_pct_last5' in df.columns and 'win_percentage' in df.columns:
        X['consistency_1'] = abs(df['win_pct_last5'] - df['win_percentage'])
        X['consistency_2'] = abs(df['win_pct_last5_2'] - df['win_percentage_2'])
        X['diff_consistency'] = X['consistency_1'] - X['consistency_2']

    if 'trend_last5' in df.columns:
        X['momentum_1'] = df['trend_last5']
        X['momentum_2'] = df['trend_last5_2']

    if 'sos_mean' in df.columns:
        X['sos_1'] = df['sos_mean']
        X['sos_2'] = df['sos_mean_2']
        if 'win_percentage' in df.columns:
            X['adj_winpct_1'] = df['win_percentage'] * (df['sos_mean'] / 1500)
            X['adj_winpct_2'] = df['win_percentage_2'] * (df['sos_mean_2'] / 1500)
            X['diff_adj_winpct'] = X['adj_winpct_1'] - X['adj_winpct_2']

    if has_massey and 'massey_avg_rank_norm' in df.columns:
        X['massey_1'] = df['massey_avg_rank_norm']
        X['massey_2'] = df['massey_avg_rank_norm_2']

    if 'seed_matchup_prior' in df.columns:
        X['seed_prior'] = df['seed_matchup_prior'].values

    return X.fillna(0)

In [5]:
def augment_features(X, y):
    X_flip = X.copy()

    diff_cols = [c for c in X.columns if c.startswith('diff_')]
    X_flip[diff_cols] = -X[diff_cols]

    if 'matchup_diff' in X_flip.columns:
        X_flip['matchup_diff'] = -X['matchup_diff']

    if 'seed_prior' in X.columns:
        X_flip['seed_prior'] = 1 - X['seed_prior']

    pairs = [
        ('seed_1', 'seed_2'), ('elo_1', 'elo_2'), ('elo_mov_1', 'elo_mov_2'),
        ('off_vs_def_1', 'off_vs_def_2'), ('consistency_1', 'consistency_2'),
        ('momentum_1', 'momentum_2'), ('sos_1', 'sos_2'),
        ('adj_winpct_1', 'adj_winpct_2'), ('massey_1', 'massey_2'),
    ]
    for a, b in pairs:
        if a in X.columns and b in X.columns:
            X_flip[a], X_flip[b] = X[b].values.copy(), X[a].values.copy()

    y_flip = 1 - y
    n = len(X)
    groups = np.concatenate([np.arange(n), np.arange(n)])

    return (pd.concat([X, X_flip], ignore_index=True),
            np.concatenate([y, y_flip]),
            groups)

In [6]:
import optuna
from sklearn.isotonic import IsotonicRegression
optuna.logging.set_verbosity(optuna.logging.WARNING)

LGB_SEEDS = [42, 123, 7]
XGB_SEEDS = [42, 123]
CAT_SEEDS = [42, 123]

def optuna_tune(X, y, groups, n_lgb=50, n_xgb=30, n_cat=30, n_lr=15):
    gkf = GroupKFold(n_splits=5)
    folds = list(gkf.split(X, y, groups))

    def lgb_obj(trial):
        p = {
            'num_leaves': trial.suggest_int('num_leaves', 6, 32),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05),
            'n_estimators': trial.suggest_int('n_estimators', 150, 800),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 0.8),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 0.9),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 40),
        }
        briers = []
        for tr_idx, val_idx in folds:
            m = lgb.LGBMClassifier(objective='binary', bagging_freq=5, verbose=-1, random_state=42, **p)
            m.fit(X.iloc[tr_idx], y[tr_idx])
            briers.append(brier_score_loss(y[val_idx], m.predict_proba(X.iloc[val_idx])[:, 1]))
        return np.mean(briers)

    lgb_study = optuna.create_study(direction='minimize')
    lgb_study.optimize(lgb_obj, n_trials=n_lgb)
    lgb_top = sorted(lgb_study.trials, key=lambda t: t.value)[:5]
    lgb_cfgs = [t.params for t in lgb_top]

    def xgb_obj(trial):
        p = {
            'max_depth': trial.suggest_int('max_depth', 2, 6),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0, log=True),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05),
            'n_estimators': trial.suggest_int('n_estimators', 150, 800),
            'subsample': trial.suggest_float('subsample', 0.5, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.8),
            'min_child_weight': trial.suggest_int('min_child_weight', 3, 20),
        }
        briers = []
        for tr_idx, val_idx in folds:
            m = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', verbosity=0, random_state=42, **p)
            m.fit(X.iloc[tr_idx], y[tr_idx])
            briers.append(brier_score_loss(y[val_idx], m.predict_proba(X.iloc[val_idx])[:, 1]))
        return np.mean(briers)

    xgb_study = optuna.create_study(direction='minimize')
    xgb_study.optimize(xgb_obj, n_trials=n_xgb)
    xgb_top = sorted(xgb_study.trials, key=lambda t: t.value)[:3]
    xgb_cfgs = [t.params for t in xgb_top]

    def cat_obj(trial):
        p = {
            'depth': trial.suggest_int('depth', 2, 6),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 10.0, log=True),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05),
            'iterations': trial.suggest_int('iterations', 150, 800),
            'subsample': trial.suggest_float('subsample', 0.5, 0.9),
            'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.4, 0.8),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 40),
        }
        briers = []
        for tr_idx, val_idx in folds:
            m = CatBoostClassifier(loss_function='Logloss', verbose=0, random_seed=42, **p)
            m.fit(X.iloc[tr_idx], y[tr_idx])
            briers.append(brier_score_loss(y[val_idx], m.predict_proba(X.iloc[val_idx])[:, 1]))
        return np.mean(briers)

    cat_study = optuna.create_study(direction='minimize')
    cat_study.optimize(cat_obj, n_trials=n_cat)
    cat_top = sorted(cat_study.trials, key=lambda t: t.value)[:3]
    cat_cfgs = [t.params for t in cat_top]

    def lr_obj(trial):
        C = trial.suggest_float('C', 0.01, 10.0, log=True)
        briers = []
        for tr_idx, val_idx in folds:
            sc = StandardScaler()
            m = LogisticRegression(C=C, max_iter=2000, random_state=42, solver='lbfgs')
            m.fit(sc.fit_transform(X.iloc[tr_idx]), y[tr_idx])
            briers.append(brier_score_loss(y[val_idx], m.predict_proba(sc.transform(X.iloc[val_idx]))[:, 1]))
        return np.mean(briers)

    lr_study = optuna.create_study(direction='minimize')
    lr_study.optimize(lr_obj, n_trials=n_lr)
    lr_top = sorted(lr_study.trials, key=lambda t: t.value)[:3]
    lr_cs = [t.params['C'] for t in lr_top]

    n_total = (len(lgb_cfgs) * len(LGB_SEEDS) + len(xgb_cfgs) * len(XGB_SEEDS) +
               len(cat_cfgs) * len(CAT_SEEDS) + len(lr_cs))
    print(f"  Optuna: LGB={lgb_study.best_value:.5f}, XGB={xgb_study.best_value:.5f}, "
          f"CAT={cat_study.best_value:.5f}, LR={lr_study.best_value:.5f}")
    print(f"  Ensemble: {len(lgb_cfgs)} LGB\u00d7{len(LGB_SEEDS)} + {len(xgb_cfgs)} XGB\u00d7{len(XGB_SEEDS)} "
          f"+ {len(cat_cfgs)} CAT\u00d7{len(CAT_SEEDS)} + {len(lr_cs)} LR = {n_total} models")

    return lgb_cfgs, xgb_cfgs, cat_cfgs, lr_cs

In [7]:
from sklearn.model_selection import GroupKFold

def _train_all_models(X_tr, y_tr, lgb_cfgs, xgb_cfgs, cat_cfgs, lr_cs):
    models = []
    for cfg in lgb_cfgs:
        for seed in LGB_SEEDS:
            m = lgb.LGBMClassifier(objective='binary', bagging_freq=5, verbose=-1, random_state=seed, **cfg)
            m.fit(X_tr, y_tr)
            models.append(('lgb', m, None))
    for cfg in xgb_cfgs:
        for seed in XGB_SEEDS:
            m = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', verbosity=0, random_state=seed, **cfg)
            m.fit(X_tr, y_tr)
            models.append(('xgb', m, None))
    for cfg in cat_cfgs:
        for seed in CAT_SEEDS:
            m = CatBoostClassifier(loss_function='Logloss', verbose=0, random_seed=seed, **cfg)
            m.fit(X_tr, y_tr)
            models.append(('cat', m, None))
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    for C in lr_cs:
        m = LogisticRegression(C=C, max_iter=2000, random_state=42, solver='lbfgs')
        m.fit(X_tr_sc, y_tr)
        models.append(('lr', m, scaler))
    return models

def _get_preds(models, X):
    preds = []
    for mtype, model, sc in models:
        if mtype == 'lr':
            preds.append(model.predict_proba(sc.transform(X))[:, 1])
        else:
            preds.append(model.predict_proba(X)[:, 1])
    return np.column_stack(preds)

def train_multi_ensemble(X_train, y_train, X_val, y_val, gender='M',
                          groups=None, lgb_cfgs=None, xgb_cfgs=None, cat_cfgs=None, lr_cs=None):
    print(f"\n  Training {gender} multi-ensemble...")
    print(f"  Train: {X_train.shape[0]} samples ({X_train.shape[0]//2} original + augmented)")
    print(f"  Features: {X_train.shape[1]}, Val: {X_val.shape[0]} samples")

    models = _train_all_models(X_train, y_train, lgb_cfgs, xgb_cfgs, cat_cfgs, lr_cs)
    n_models = len(models)
    print(f"  {n_models} models trained (LGB:{len(lgb_cfgs)*len(LGB_SEEDS)}, "
          f"XGB:{len(xgb_cfgs)*len(XGB_SEEDS)}, CAT:{len(cat_cfgs)*len(CAT_SEEDS)}, LR:{len(lr_cs)})")

    print(f"  Running 5-fold CV for OOF predictions...")
    gkf = GroupKFold(n_splits=5)
    oof = np.zeros((len(X_train), n_models))

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
        fold_models = _train_all_models(X_train.iloc[tr_idx], y_train[tr_idx], lgb_cfgs, xgb_cfgs, cat_cfgs, lr_cs)
        oof[val_idx] = _get_preds(fold_models, X_train.iloc[val_idx])

    oof_briers = [brier_score_loss(y_train, oof[:, i]) for i in range(n_models)]
    inv_weights = np.array([1.0 / b for b in oof_briers])
    inv_weights /= inv_weights.sum()

    meta_model = LogisticRegression(C=1.0, max_iter=2000, random_state=42)
    meta_model.fit(oof, y_train)

    oof_weighted = oof @ inv_weights
    oof_stacked = meta_model.predict_proba(oof)[:, 1]

    calibrator_w = IsotonicRegression(out_of_bounds='clip')
    calibrator_w.fit(oof_weighted, y_train)
    calibrator_s = IsotonicRegression(out_of_bounds='clip')
    calibrator_s.fit(oof_stacked, y_train)

    print(f"  OOF Brier (weighted):     {brier_score_loss(y_train, oof_weighted):.5f}")
    print(f"  OOF Brier (stacking):     {brier_score_loss(y_train, oof_stacked):.5f}")
    print(f"  OOF Brier (w+calibrated): {brier_score_loss(y_train, calibrator_w.predict(oof_weighted)):.5f}")
    print(f"  OOF Brier (s+calibrated): {brier_score_loss(y_train, calibrator_s.predict(oof_stacked)):.5f}")

    val_matrix = _get_preds(models, X_val)
    val_w = val_matrix @ inv_weights
    val_s = meta_model.predict_proba(val_matrix)[:, 1]
    val_wc = calibrator_w.predict(val_w)
    val_sc = calibrator_s.predict(val_s)

    print(f"  Val Brier (weighted):     {brier_score_loss(y_val, val_w):.5f}")
    print(f"  Val Brier (stacking):     {brier_score_loss(y_val, val_s):.5f}")
    print(f"  Val Brier (w+calibrated): {brier_score_loss(y_val, val_wc):.5f}")
    print(f"  Val Brier (s+calibrated): {brier_score_loss(y_val, val_sc):.5f}")
    print(f"  Best individual OOF:      {min(oof_briers):.5f}")

    return models, inv_weights, meta_model, calibrator_w, calibrator_s

def predict_multi_ensemble(models, weights, X, meta_model=None, calibrator=None):
    pred_matrix = _get_preds(models, X)

    if meta_model is not None:
        pred = meta_model.predict_proba(pred_matrix)[:, 1]
    else:
        pred = pred_matrix @ weights

    if calibrator is not None:
        pred = calibrator.predict(pred)

    return np.clip(pred, 0.01, 0.99)

In [8]:
def select_features(X_train, y_train, groups, lgb_cfgs, threshold_pct=10):
    m = lgb.LGBMClassifier(objective='binary', bagging_freq=5, verbose=-1, random_state=42, **lgb_cfgs[0])
    m.fit(X_train, y_train)
    
    imp = pd.Series(m.feature_importances_, index=X_train.columns)
    threshold = np.percentile(imp.values, threshold_pct)
    keep = imp[imp > threshold].index.tolist()
    drop = imp[imp <= threshold].index.tolist()
    
    if 'seed_prior' in drop:
        drop.remove('seed_prior')
        keep.append('seed_prior')
    
    print(f"  Feature selection: {len(X_train.columns)} -> {len(keep)} features (dropped {len(drop)})")
    if drop:
        print(f"    Dropped: {', '.join(drop[:10])}{'...' if len(drop) > 10 else ''}")
    return keep

def cv_optimize_blend_alpha(X_train, y_train, groups, models, weights, meta, cal,
                             seed_prior_col='seed_prior', alphas=None):
    if alphas is None:
        alphas = np.arange(0.5, 1.01, 0.025)
    
    gkf = GroupKFold(n_splits=5)
    
    oof_preds = np.zeros(len(X_train))
    for tr_idx, val_idx in gkf.split(X_train, y_train, groups):
        fold_preds = predict_multi_ensemble(models, weights, X_train.iloc[val_idx], meta, cal)
        oof_preds[val_idx] = fold_preds
    
    sp_train = X_train[seed_prior_col].values if seed_prior_col in X_train.columns else np.full(len(X_train), 0.5)
    
    best_alpha, best_brier = 1.0, 999
    for a in alphas:
        blended = a * oof_preds + (1 - a) * sp_train
        b = brier_score_loss(y_train, np.clip(blended, 0.01, 0.99))
        if b < best_brier:
            best_alpha, best_brier = a, b
    
    return best_alpha, best_brier

def optimize_clipping(preds, y_true, clips=None):
    if clips is None:
        clips = [0.01, 0.02, 0.03, 0.04, 0.05]
    
    best_clip, best_brier = 0.01, 999
    for lo in clips:
        clipped = np.clip(preds, lo, 1 - lo)
        b = brier_score_loss(y_true, clipped)
        if b < best_brier:
            best_clip, best_brier = lo, b
    return best_clip

def run_pipeline(gender='M'):
    prefix = 'm' if gender == 'M' else 'w'
    label = 'MEN' if gender == 'M' else 'WOMEN'
    has_massey = (gender == 'M')

    print(f"\n{'='*60}")
    print(f"  {label} MODEL")
    print(f"{'='*60}")

    train = pd.read_csv(f'tourney_train_{prefix}.csv')
    val = pd.read_csv(f'tourney_val_{prefix}.csv')
    test = pd.read_csv(f'tourney_test_{prefix}.csv')
    sf = pd.read_csv(f'season_features_{prefix}.csv')

    train = enrich_tourney_data(train, gender)
    val = enrich_tourney_data(val, gender)
    test = enrich_tourney_data(test, gender)

    print(f"  Raw: Train {train.shape[0]}, Val {val.shape[0]}, Test {test.shape[0]}")

    X_train_raw = prepare_features(train, has_massey=has_massey)
    y_train_raw = train['Is1Winner'].values
    X_val = prepare_features(val, has_massey=has_massey)
    y_val = val['Is1Winner'].values
    X_test = prepare_features(test, has_massey=has_massey)
    y_test = test['Is1Winner'].values

    sp_val = X_val['seed_prior'].values.copy() if 'seed_prior' in X_val.columns else np.full(len(X_val), 0.5)
    sp_test = X_test['seed_prior'].values.copy() if 'seed_prior' in X_test.columns else np.full(len(X_test), 0.5)

    X_train, y_train, groups = augment_features(X_train_raw, y_train_raw)
    print(f"  After augmentation: {X_train.shape[0]} samples, {X_train.shape[1]} features")

    for c in X_train.columns:
        if c not in X_val.columns: X_val[c] = 0
        if c not in X_test.columns: X_test[c] = 0
    X_val = X_val[X_train.columns]
    X_test = X_test[X_train.columns]

    print(f"\n  Running Optuna tuning...")
    lgb_cfgs, xgb_cfgs, cat_cfgs, lr_cs = optuna_tune(X_train, y_train, groups)

    keep_cols = select_features(X_train, y_train, groups, lgb_cfgs, threshold_pct=10)
    X_train = X_train[keep_cols]
    X_val = X_val[keep_cols]
    X_test = X_test[keep_cols]

    models, weights, meta, cal_w, cal_s = train_multi_ensemble(
        X_train, y_train, X_val, y_val, gender, groups=groups,
        lgb_cfgs=lgb_cfgs, xgb_cfgs=xgb_cfgs, cat_cfgs=cat_cfgs, lr_cs=lr_cs)

    methods = {
        'weighted':     (None, None),
        'stacking':     (meta, None),
        'w+calibrated': (None, cal_w),
        's+calibrated': (meta, cal_s),
    }

    print(f"\n  {'Method':20s} {'Val(2025)':>10s} {'Test(2024)':>10s} {'Avg':>10s}")
    print(f"  {'---'*18}")

    best_name, best_avg = None, 999
    results = {}
    for name, (m, c) in methods.items():
        v_pred = predict_multi_ensemble(models, weights, X_val, m, c)
        t_pred = predict_multi_ensemble(models, weights, X_test, m, c)
        v = brier_score_loss(y_val, v_pred)
        t = brier_score_loss(y_test, t_pred)
        avg = (v + t) / 2
        print(f"  {name:20s} {v:10.5f} {t:10.5f} {avg:10.5f}")
        results[name] = (v, t, avg, m, c, v_pred, t_pred)
        if avg < best_avg:
            best_avg = avg
            best_name = name

    best_meta, best_cal = results[best_name][3], results[best_name][4]
    best_val_pred = results[best_name][5]
    best_test_pred = results[best_name][6]

    print(f"\n  Blending with seed priors (CV-optimized)...")
    blend_alpha, cv_blend_brier = cv_optimize_blend_alpha(
        X_train, y_train, groups, models, weights, best_meta, best_cal)
    
    blended_val = blend_alpha * best_val_pred + (1 - blend_alpha) * sp_val
    blended_test = blend_alpha * best_test_pred + (1 - blend_alpha) * sp_test
    bv = brier_score_loss(y_val, np.clip(blended_val, 0.01, 0.99))
    bt = brier_score_loss(y_test, np.clip(blended_test, 0.01, 0.99))
    
    unblended_v = results[best_name][0]
    unblended_t = results[best_name][1]
    print(f"  CV blend alpha={blend_alpha:.3f} (CV Brier={cv_blend_brier:.5f})")
    print(f"  Before blend: Val={unblended_v:.5f}, Test={unblended_t:.5f}")
    print(f"  After blend:  Val={bv:.5f}, Test={bt:.5f}")
    
    use_blend = ((bv + bt) / 2) < ((unblended_v + unblended_t) / 2)
    if use_blend:
        val_brier, test_brier = bv, bt
        print(f"  -> Using blended predictions (alpha={blend_alpha:.3f})")
    else:
        val_brier, test_brier = unblended_v, unblended_t
        blend_alpha = 1.0
        print(f"  -> Blending didn't help, using pure model predictions")

    print(f"\n  Optimizing clipping range...")
    if use_blend:
        clip_preds_val = blended_val
        clip_preds_test = blended_test
    else:
        clip_preds_val = best_val_pred
        clip_preds_test = best_test_pred
    
    best_clip = optimize_clipping(clip_preds_val, y_val)
    clipped_val = np.clip(clip_preds_val, best_clip, 1 - best_clip)
    clipped_test = np.clip(clip_preds_test, best_clip, 1 - best_clip)
    cv_clip = brier_score_loss(y_val, clipped_val)
    ct_clip = brier_score_loss(y_test, clipped_test)
    
    print(f"  Best clip=[{best_clip:.2f}, {1-best_clip:.2f}]")
    print(f"  Before clip: Val={val_brier:.5f}, Test={test_brier:.5f}")
    print(f"  After clip:  Val={cv_clip:.5f}, Test={ct_clip:.5f}")
    
    if ((cv_clip + ct_clip) / 2) < ((val_brier + test_brier) / 2):
        val_brier, test_brier = cv_clip, ct_clip
        print(f"  -> Using clipping [{best_clip:.2f}, {1-best_clip:.2f}]")
    else:
        best_clip = 0.01
        print(f"  -> Clipping didn't help, keeping default [0.01, 0.99]")

    print(f"\n  -> Best: {best_name} {'+ blend' if use_blend else ''} (val={val_brier:.5f}, test={test_brier:.5f})")

    X_val_aug, y_val_aug, _ = augment_features(X_val, y_val)
    X_test_aug, y_test_aug, _ = augment_features(X_test, y_test)
    X_all = pd.concat([X_train, X_val_aug, X_test_aug], ignore_index=True)
    y_all = np.concatenate([y_train, y_val_aug, y_test_aug])

    print(f"\n  Retraining on full augmented data ({X_all.shape[0]} samples)...")
    final_models = _train_all_models(X_all, y_all, lgb_cfgs, xgb_cfgs, cat_cfgs, lr_cs)

    imp = pd.Series(
        final_models[0][1].feature_importances_, index=X_train.columns
    ).sort_values(ascending=False)
    print(f"\n  Top 15 features ({label}):")
    for feat, score in imp.head(15).items():
        print(f"    {feat:45s} {score:6d}")

    return final_models, weights, best_meta, best_cal, X_train.columns.tolist(), sf, test_brier, val_brier, blend_alpha, best_clip

In [9]:
def generate_2026_predictions(models, weights, meta, cal, feature_cols, sf, gender='M', blend_alpha=1.0, clip_lo=0.01):
    label = 'MEN' if gender == 'M' else 'WOMEN'
    has_massey = (gender == 'M')
    seed_priors = seed_priors_m if gender == 'M' else seed_priors_w

    sub = pd.read_csv(DATA_DIR / 'SampleSubmissionStage2.csv')
    if gender == 'M':
        sub = sub[sub['ID'].apply(lambda x: int(x.split('_')[1]) < 3000)].copy()
    else:
        sub = sub[sub['ID'].apply(lambda x: int(x.split('_')[1]) >= 3000)].copy()

    method = 'stacking' if meta else 'weighted'
    if cal: method += '+calibrated'
    if blend_alpha < 1.0: method += f'+blend({blend_alpha:.2f})'
    method += f'+clip[{clip_lo:.2f},{1-clip_lo:.2f}]'
    print(f"\n  Generating {len(sub):,} predictions for {label} 2026 [{method}]")

    sf_2026 = sf[sf['Season'] == 2026].copy()
    sf_2026['Seed'] = sf_2026['TeamID'].apply(lambda t: seeds_lookup.get((2026, t), 0))
    print(f"  Teams: {len(sf_2026)}, Seeded: {(sf_2026['Seed'] > 0).sum()}")

    sos = sos_m if gender == 'M' else sos_w
    sf_2026 = sf_2026.merge(sos[sos['Season'] == 2026][['TeamID', 'sos_mean', 'sos_max']], on='TeamID', how='left')
    sf_2026[['sos_mean', 'sos_max']] = sf_2026[['sos_mean', 'sos_max']].fillna(1500)

    if gender == 'M':
        sf_2026 = sf_2026.merge(massey_comp[massey_comp['Season'] == 2026].drop(columns='Season'), on='TeamID', how='left')
        sf_2026[['massey_avg_rank_norm', 'massey_best_rank_norm']] = sf_2026[['massey_avg_rank_norm', 'massey_best_rank_norm']].fillna(0)

    feat_cols_sf = [c for c in sf_2026.columns if c not in ['Season', 'TeamID']]
    sf_dict = {int(r['TeamID']): r[feat_cols_sf].to_dict() for _, r in sf_2026.iterrows()}

    seed_prior_preds = []
    rows = []
    for _, r in sub.iterrows():
        t1, t2 = int(r['ID'].split('_')[1]), int(r['ID'].split('_')[2])
        
        s1 = raw_seeds_lookup.get((2026, t1))
        s2 = raw_seeds_lookup.get((2026, t2))
        sp = seed_priors.get((s1, s2), 0.5) if s1 and s2 else 0.5
        seed_prior_preds.append(sp)
        
        if t1 in sf_dict and t2 in sf_dict:
            row = {}
            for c in feat_cols_sf:
                row[c] = sf_dict[t1][c]
                row[c + '_2'] = sf_dict[t2][c]
            row['seed_matchup_prior'] = sp
            rows.append(row)
        else:
            row = {c: 0 for c in feat_cols_sf}
            row.update({c + '_2': 0 for c in feat_cols_sf})
            row['seed_matchup_prior'] = sp
            rows.append(row)

    seed_prior_preds = np.array(seed_prior_preds)

    X_pred = prepare_features(pd.DataFrame(rows), has_massey=has_massey)
    for c in feature_cols:
        if c not in X_pred.columns: X_pred[c] = 0
    X_pred = X_pred[feature_cols]

    model_preds = predict_multi_ensemble(models, weights, X_pred, meta, cal)
    
    if blend_alpha < 1.0:
        preds = blend_alpha * model_preds + (1 - blend_alpha) * seed_prior_preds
    else:
        preds = model_preds
    
    preds = np.clip(preds, clip_lo, 1 - clip_lo)
    
    sub['Pred'] = preds
    print(f"  Predictions: mean={preds.mean():.4f}, std={preds.std():.4f}, range=[{preds.min():.4f}, {preds.max():.4f}]")

    return sub

In [10]:
m_models, m_weights, m_meta, m_cal, m_cols, m_sf, m_test_b, m_val_b, m_alpha, m_clip = run_pipeline('M')
m_preds = generate_2026_predictions(m_models, m_weights, m_meta, m_cal, m_cols, m_sf, 'M', m_alpha, m_clip)


  MEN MODEL
  Raw: Train 1315, Val 67, Test 67
  After augmentation: 2630 samples, 70 features

  Running Optuna tuning...
  Optuna: LGB=0.18799, XGB=0.18731, CAT=0.18620, LR=0.18740
  Ensemble: 5 LGB×3 + 3 XGB×2 + 3 CAT×2 + 3 LR = 30 models
  Feature selection: 70 -> 63 features (dropped 7)
    Dropped: diff_win_percentage, diff_win_pct_last5, diff_win_pct_last10, diff_Seed, seed_1, seed_2, seed_sum

  Training M multi-ensemble...
  Train: 2630 samples (1315 original + augmented)
  Features: 63, Val: 67 samples
  30 models trained (LGB:15, XGB:6, CAT:6, LR:3)
  Running 5-fold CV for OOF predictions...
  OOF Brier (weighted):     0.18633
  OOF Brier (stacking):     0.18631
  OOF Brier (w+calibrated): 0.18225
  OOF Brier (s+calibrated): 0.18069
  Val Brier (weighted):     0.15003
  Val Brier (stacking):     0.14528
  Val Brier (w+calibrated): 0.14978
  Val Brier (s+calibrated): 0.14580
  Best individual OOF:      0.18605

  Method                Val(2025) Test(2024)        Avg
  ------

In [11]:
w_models, w_weights, w_meta, w_cal, w_cols, w_sf, w_test_b, w_val_b, w_alpha, w_clip = run_pipeline('W')
w_preds = generate_2026_predictions(w_models, w_weights, w_meta, w_cal, w_cols, w_sf, 'W', w_alpha, w_clip)


  WOMEN MODEL
  Raw: Train 827, Val 67, Test 67
  After augmentation: 1654 samples, 66 features

  Running Optuna tuning...
  Optuna: LGB=0.14670, XGB=0.14615, CAT=0.14500, LR=0.14079
  Ensemble: 5 LGB×3 + 3 XGB×2 + 3 CAT×2 + 3 LR = 30 models
  Feature selection: 66 -> 59 features (dropped 7)
    Dropped: diff_tempo, diff_win_pct_last10, diff_off_rating_last5, diff_def_rating_last10, seed_sum, tempo_avg, tempo_mismatch

  Training W multi-ensemble...
  Train: 1654 samples (827 original + augmented)
  Features: 59, Val: 67 samples
  30 models trained (LGB:15, XGB:6, CAT:6, LR:3)
  Running 5-fold CV for OOF predictions...
  OOF Brier (weighted):     0.14495
  OOF Brier (stacking):     0.13994
  OOF Brier (w+calibrated): 0.14137
  OOF Brier (s+calibrated): 0.13489
  Val Brier (weighted):     0.10195
  Val Brier (stacking):     0.10347
  Val Brier (w+calibrated): 0.10781
  Val Brier (s+calibrated): 0.10877
  Best individual OOF:      0.14081

  Method                Val(2025) Test(2024)  

In [12]:
submission = pd.concat([m_preds, w_preds], ignore_index=True)[['ID', 'Pred']]
submission.to_csv('submission_rys.csv', index=False)

print(f"{'='*60}")
print(f"  RESULTS SUMMARY")
print(f"{'='*60}")
print(f"  Men   - Val(2025): {m_val_b:.5f}, Test(2024): {m_test_b:.5f}")
print(f"  Women - Val(2025): {w_val_b:.5f}, Test(2024): {w_test_b:.5f}")
print(f"  Submission: {len(submission):,} rows -> submission_rys.csv")
print(f"  Men: {len(m_preds):,}, Women: {len(w_preds):,}")

expected = pd.read_csv(DATA_DIR / 'SampleSubmissionStage2.csv')
assert len(submission) == len(expected), f"Row mismatch: {len(submission)} vs {len(expected)}"
assert set(submission['ID']) == set(expected['ID']), "ID mismatch!"
assert submission['Pred'].between(0, 1).all(), "Predictions out of [0,1]!"
print("  Format verification: PASSED")

  RESULTS SUMMARY
  Men   - Val(2025): 0.14580, Test(2024): 0.18782
  Women - Val(2025): 0.10195, Test(2024): 0.12530
  Submission: 132,133 rows -> submission_rys.csv
  Men: 66,430, Women: 65,703
  Format verification: PASSED
